# Lire l’évolution mensuelle

Une série temporelle exige des dates réelles, un calendrier trié et une agrégation mensuelle avant le tracé.

**Question métier :** Les ventes augmentent-elles à chaque mois de la période ?

Les données sont montées en lecture seule. Enregistrez vos figures dans `/workspace/resultats`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA = '/workspace/data/sample_-_superstore.xls'
orders = pd.read_excel(DATA, sheet_name='Orders', engine='xlrd')
orders['Order Date'] = pd.to_datetime(orders['Order Date'])

## Notion 1 — Agréger les ventes par mois


### Étape 1 — Utiliser une date comme index

resample crée des périodes mensuelles sur un index datetime. Une fois les mois calculés, loc["2023"] sélectionne les valeurs de cette année.

In [ ]:
mensuel = orders.set_index("Order Date").resample("MS")["Sales"].sum()
print(mensuel.loc["2023"])

### Étape 2 — Tracer la série chronologique

Une courbe convient à une mesure ordonnée dans le temps.

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(mensuel.index, mensuel.values, marker='o')
ax.set(title='Ventes mensuelles', ylabel='Ventes (unités monétaires)')
ax.grid(axis='y', alpha=.25)
plt.show()

### Étape 3 — Interpréter

La série contient 48 mois consécutifs. Le tracé montre des variations mensuelles ; il ne permet pas d’en déduire une cause.

### Exercice 1 — Étudier les commandes mensuelles de 2026

Quel mois de 2026 compte le plus de commandes distinctes ? Construisez le graphique du nombre de commandes pour les douze mois de cette année.

**À rendre :** Un graphique mensuel, le mois recherché et son nombre de commandes distinctes.

In [ ]:
commandes_mois = orders.set_index("Order Date").resample("MS")["Order ID"].nunique()
commandes_2026 = commandes_mois.loc["2026"]
print(commandes_2026)
print("Mois au plus haut :", commandes_2026.idxmax(), commandes_2026.max())
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(commandes_2026.index, commandes_2026.values, width=20, color="#275c47")
ax.set(title="Commandes distinctes par mois en 2026", ylabel="Nombre de commandes")
plt.tight_layout()
plt.show()

### Mise en commun

Novembre 2026 compte le plus de commandes distinctes : 262. Le comptage porte sur les identifiants Order ID distincts de chaque mois, et non sur le nombre de lignes.

## Notion 2 — Calculer une variation en pourcentage


### Étape 1 — Comparer deux périodes

pct_change mesure une variation relative par rapport à la période précédente.

In [ ]:
variation_pct = mensuel.pct_change() * 100
print(variation_pct.head())

### Étape 2 — Afficher les hausses et les baisses

La ligne zéro sépare les variations positives des négatives.

In [ ]:
variation_pct = variation_pct.dropna()
fig, ax = plt.subplots(figsize=(10,4))
ax.bar(variation_pct.index, variation_pct.values, width=20)
ax.axhline(0, color='black', linewidth=1)
ax.set(title='Variation relative des ventes', ylabel='Variation (%)')
plt.show()

### Étape 3 — Interpréter

pct_change renvoie une proportion ; ×100 l’exprime en pourcentage. Le premier mois est NaN, car aucune période précédente ne sert de référence.

### Exercice 2 — Repérer les hausses et les baisses

Quel mois présente la plus forte hausse relative ? Quel mois présente la plus forte baisse relative ?

**À rendre :** Les deux dates, leurs pourcentages et la référence utilisée.

In [ ]:
variation_pct = mensuel.pct_change() * 100
print(variation_pct.idxmax(), variation_pct.max())
print(variation_pct.idxmin(), variation_pct.min())

### Mise en commun

Les pourcentages dépendent de la base du mois précédent. Une hausse élevée en pourcentage peut porter sur un montant modeste.

## Notion 3 — Calculer une variation en montant


### Étape 1 — Soustraire le mois précédent

diff mesure le changement dans l’unité originale de Sales.

In [ ]:
variation_montant = mensuel.diff()
print(variation_montant.head())

### Étape 2 — Tracer les écarts

La même ligne zéro distingue les augmentations des diminutions.

In [ ]:
variation_montant = variation_montant.dropna()
fig, ax = plt.subplots(figsize=(10,4))
ax.bar(variation_montant.index, variation_montant.values, width=20)
ax.axhline(0, color='black', linewidth=1)
ax.set(title='Écart de ventes d’un mois à l’autre', ylabel='Unités monétaires')
plt.show()

### Étape 3 — Interpréter

diff et pct_change utilisent la même période de référence, mais diff s’exprime en unités monétaires alors que pct_change est relatif.

### Exercice 3 — Comparer deux façons de mesurer la hausse

Le mois de plus forte hausse en montant est-il aussi celui de plus forte hausse en pourcentage ?

**À rendre :** Les deux mois, leurs variations et une phrase qui compare leurs unités.

In [ ]:
variation_montant = mensuel.diff()
variation_pct = mensuel.pct_change() * 100
print(variation_montant.idxmax(), variation_montant.max())
print(variation_pct.idxmax(), variation_pct.max())

### Mise en commun

Les maxima peuvent survenir à des dates différentes : l’un mesure un montant ajouté, l’autre un taux rapporté au mois précédent.